# Geopolitical Exposure Proof of Concept
## 03 — Outcome-Free Embedding Expansion

This notebook propagates the frozen text-based exposure seeds through five pre-event holdings embeddings. All scores, comparison models, and rankings are frozen without loading event returns. The primary comparison is text-only direct exposure versus text-seeded embedding propagation.

## 1. Environment and immutable inputs

The notebook requires the hashed seed manifest produced by Notebook 02 and the five 2021Q4 Binary embedding representations.

In [ ]:
from pathlib import Path
import hashlib, importlib, json, sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'geopolitical_exposure':
    GEO_ROOT, ROOT = ROOT, ROOT.parent
else:
    GEO_ROOT = ROOT / 'geopolitical_exposure'
if not GEO_ROOT.exists():
    raise FileNotFoundError('Open from Asset_Embeddings or geopolitical_exposure.')
sys.path.insert(0, str(GEO_ROOT / 'src'))
import embedding_expansion
importlib.reload(embedding_expansion)
from embedding_expansion import l2_normalize, propagate_all_channels, propagate_channel, standardized_characteristics

PROCESSED = GEO_ROOT / 'documents' / 'processed'
RESULTS = GEO_ROOT / 'results'
MANIFESTS = GEO_ROOT / 'manifests'
EXPANSION = GEO_ROOT / 'embedding_expansion'
EXPANSION.mkdir(parents=True, exist_ok=True)
OUTCOMES_LOADED = False
print('Asset project:', ROOT)
print('Event outcomes loaded:', OUTCOMES_LOADED)

## 2. Verify the frozen seed hash

Any change to a firm, channel, direction, score, evidence quote, or source-document hash invalidates this handoff.

In [ ]:
seeds = pd.read_parquet(PROCESSED / 'text_exposure_seeds_frozen_v1.parquet')
with open(MANIFESTS / 'text_exposure_seed_freeze_v1.json') as handle:
    seed_manifest = json.load(handle)
seed_hash_columns = ['firm_id', 'channel', 'direction', 'signed_score', 'confidence',
                     'canonical_evidence', 'source_content_sha256']
seed_payload = seeds[seed_hash_columns].sort_values(['firm_id', 'channel']).to_dict('records')
recomputed_seed_hash = hashlib.sha256(
    json.dumps(seed_payload, sort_keys=True, default=str).encode()
).hexdigest()
seed_check = pd.DataFrame([{
    'manifest_seed_sha256': seed_manifest['seed_sha256'],
    'recomputed_seed_sha256': recomputed_seed_hash,
    'hash_matches': recomputed_seed_hash == seed_manifest['seed_sha256'],
    'seed_firms': seeds.firm_id.nunique(), 'firm_channel_seeds': len(seeds),
    'outcomes_inspected_when_frozen': seed_manifest['outcomes_inspected'],
}])
display(seed_check)
assert seed_check.hash_matches.all() and not seed_manifest['outcomes_inspected']

## 3. Load and align the five pre-event embedding representations

All five representations must contain the same 1,805 PERMCOs and every accepted seed. Cosine similarity is invariant to arbitrary orthogonal rotations of the embedding coordinates.

In [ ]:
embedding_paths = {
    'all_s12': ROOT / 'active_passive_extension/benchmarks/all_s12_control/2021q4/rs_binary/asset_embeddings_10d.parquet',
    'stable_stock_picker': ROOT / 'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/stable_stock_picker/rs_binary/asset_embeddings_10d.parquet',
    'tactical_active': ROOT / 'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/tactical_active/rs_binary/asset_embeddings_10d.parquet',
    'benchmark_rebalancer': ROOT / 'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/benchmark_rebalancer/rs_binary/asset_embeddings_10d.parquet',
    'passive_like': ROOT / 'active_passive_extension/benchmarks/behavioral_embeddings/2021q4/passive_like/rs_binary/asset_embeddings_10d.parquet',
}
embedding_frames = {}
audit_rows = []
reference_firms = None
for name, path in embedding_paths.items():
    frame = pd.read_parquet(path).sort_values('permco').reset_index(drop=True)
    dimensions = [column for column in frame if column.startswith('embedding_')]
    firms = tuple(frame.permco.astype(int))
    reference_firms = firms if reference_firms is None else reference_firms
    embedding_frames[name] = frame
    audit_rows.append({
        'representation': name, 'firms': len(frame), 'dimensions': len(dimensions),
        'duplicate_firms': int(frame.permco.duplicated().sum()),
        'same_ordered_universe': firms == reference_firms,
        'seed_firms_present': int(seeds.firm_id.isin(frame.permco).sum()),
    })
embedding_audit = pd.DataFrame(audit_rows)
display(embedding_audit)
assert embedding_audit[['duplicate_firms']].eq(0).all().all()
assert embedding_audit.same_ordered_universe.all()
assert embedding_audit.seed_firms_present.eq(len(seeds)).all()

## 4. Freeze the comparison design

Embeddings alone provide similarity but no geopolitical sign. The clean primary comparison therefore asks whether propagating independently frozen text labels through holdings similarity adds information beyond text-only labels. Characteristics and shuffled embeddings use the same seeds, preventing an outcome-selected benchmark.

In [ ]:
comparison_design = pd.DataFrame([
    {'model': 'text_only', 'inputs': 'Accepted pre-event text seeds; zero for non-seeds',
     'role': 'Direct-disclosure baseline'},
    {'model': 'all_s12_embedding', 'inputs': 'Text seeds propagated through all-S12 cosine similarity',
     'role': 'Primary embedding expansion'},
    {'model': 'behavioral_embeddings', 'inputs': 'Same seeds through four investor-behavior embeddings',
     'role': 'Which investor behavior carries useful relationships'},
    {'model': 'characteristics_similarity', 'inputs': 'Same seeds through standardized firm characteristics',
     'role': 'Conventional observable-firm baseline'},
    {'model': 'shuffled_embedding_placebo', 'inputs': 'Same seed weights assigned to random firms',
     'role': 'Null distribution for accidental embedding structure'},
    {'model': 'industry_baseline', 'inputs': 'Pre-event industry codes acquired before evaluation',
     'role': 'Test whether embeddings only rediscover industries'},
])
display(comparison_design)
design_payload = {
    'seed_sha256': seed_manifest['seed_sha256'], 'embedding_date': '2021Q4',
    'similarity': 'row_l2_normalized_cosine',
    'propagation': 'sum(cosine * signed_score * confidence) / sum(abs(signed_score * confidence))',
    'seed_self_evaluation': False, 'channel_specific': True,
    'placebo_repetitions': 100, 'placebo_seed': 20220224,
    'outcomes_loaded': False, 'models': comparison_design.to_dict('records'),
}
design_sha256 = hashlib.sha256(json.dumps(design_payload, sort_keys=True).encode()).hexdigest()
design_payload['sha256'] = design_sha256
(MANIFESTS / 'embedding_expansion_design_v1.json').write_text(json.dumps(design_payload, indent=2))
print('Expansion design SHA-256:', design_sha256)

## 5. Construct the text-only benchmark

Text-only exposure is nonzero only for accepted direct seeds. Direct seeds will be excluded from the main non-seed event evaluation.

In [ ]:
universe = embedding_frames['all_s12'][['permco', 'firm_name']].copy()
channels = sorted(seeds.channel.unique())
text_rows = []
for channel in channels:
    frame = universe.copy()
    frame['representation'] = 'text_only'
    frame['channel'] = channel
    channel_scores = seeds.loc[seeds.channel.eq(channel)].set_index('firm_id')['signed_score']
    frame['text_score'] = frame.permco.map(channel_scores).fillna(0.0)
    frame['is_seed'] = frame.permco.isin(seeds.firm_id)
    text_rows.append(frame)
text_scores = pd.concat(text_rows, ignore_index=True)
text_scores.to_parquet(EXPANSION / 'text_only_scores.parquet', index=False)
display(text_scores.groupby('channel').agg(firms=('permco', 'size'),
                                                 nonzero_scores=('text_score', lambda x: int(x.ne(0).sum()))).reset_index())

## 6. Propagate seeds through each holdings embedding

Scores are constructed separately by channel. Seed firms remain in the saved panel for auditing but are flagged and excluded from downstream expansion tests.

In [ ]:
propagation_frames = []
for name, frame in embedding_frames.items():
    dimensions = [column for column in frame if column.startswith('embedding_')]
    scores = propagate_all_channels(frame, dimensions, seeds, name)
    propagation_frames.append(scores)
    print(name, 'complete')
embedding_scores = pd.concat(propagation_frames, ignore_index=True)
embedding_scores.to_parquet(EXPANSION / 'embedding_propagated_scores.parquet', index=False)
propagation_audit = (embedding_scores.groupby(['representation', 'channel'], as_index=False)
                     .agg(firms=('permco', 'size'), seeds=('is_seed', 'sum'),
                          score_min=('propagated_score', 'min'),
                          score_median=('propagated_score', 'median'),
                          score_max=('propagated_score', 'max')))
display(propagation_audit)

## 7. Construct the characteristics-similarity baseline

Pre-event beta, size, book-to-market, investment, profitability, and momentum are median-imputed and standardized without event outcomes. The identical signed seeds are then propagated through characteristic cosine similarity.

In [ ]:
controls_path = ROOT / 'remaining_benchmarks/return_comovement/characteristics/2021q4.parquet'
controls = pd.read_parquet(controls_path)
characteristic_columns = ['beta_60m_paper', 'log_market_equity', 'log_book_to_market',
                          'at_gr1_paper', 'gp_at_paper', 'momentum_12_1']
characteristic_frame = universe.merge(controls, on='permco', how='left', validate='one_to_one')
characteristic_matrix = standardized_characteristics(characteristic_frame, characteristic_columns)
characteristic_scores = pd.concat([
    propagate_channel(characteristic_frame, characteristic_matrix, group, 'characteristics_similarity')
    for _, group in seeds.groupby('channel', sort=True)
], ignore_index=True)
characteristic_scores.to_parquet(EXPANSION / 'characteristic_propagated_scores.parquet', index=False)
coverage = characteristic_frame[characteristic_columns].notna().mean().rename('coverage').reset_index()
coverage = coverage.rename(columns={'index': 'characteristic'})
display(coverage)

## 8. Build 100 shuffled-seed placebo expansions

Each repetition preserves channel seed counts and weights but assigns them to random non-seed firms. This creates a frozen null distribution without touching outcomes.

In [ ]:
rng = np.random.default_rng(20220224)
base_frame = embedding_frames['all_s12']
dimensions = [column for column in base_frame if column.startswith('embedding_')]
base_matrix = l2_normalize(base_frame[dimensions].to_numpy())
eligible_placebo_firms = base_frame.loc[~base_frame.permco.isin(seeds.firm_id), 'permco'].to_numpy()
placebo_frames = []
for repetition in range(100):
    for channel, group in seeds.groupby('channel', sort=True):
        pseudo = group.copy()
        pseudo['firm_id'] = rng.choice(eligible_placebo_firms, size=len(group), replace=False)
        scores = propagate_channel(base_frame, base_matrix, pseudo, 'shuffled_embedding_placebo')
        scores['repetition'] = repetition
        placebo_frames.append(scores[['repetition', 'permco', 'channel', 'propagated_score']])
placebo_scores = pd.concat(placebo_frames, ignore_index=True)
placebo_scores.to_parquet(EXPANSION / 'shuffled_embedding_placebos_100.parquet', index=False)
display(pd.DataFrame([{'repetitions': placebo_scores.repetition.nunique(),
                       'rows': len(placebo_scores), 'channels': placebo_scores.channel.nunique()}]))

## 9. Compare representations before outcomes

Rank correlations and top non-seed candidates show whether behavioral investor groups produce meaningfully different expansions. These diagnostics do not determine which model wins; that decision is reserved for the frozen event evaluation.

In [ ]:
nonseed_scores = embedding_scores.loc[~embedding_scores.is_seed].copy()
correlation_rows = []
for channel, group in nonseed_scores.groupby('channel'):
    wide = group.pivot(index='permco', columns='representation', values='propagated_score')
    correlation = wide.corr(method='spearman')
    for left in correlation.columns:
        for right in correlation.columns:
            if left < right:
                correlation_rows.append({'channel': channel, 'left': left, 'right': right,
                                         'spearman_correlation': correlation.loc[left, right]})
representation_correlations = pd.DataFrame(correlation_rows)
representation_correlations.to_parquet(EXPANSION / 'representation_rank_correlations.parquet', index=False)
top_candidates = (nonseed_scores.sort_values(
    ['representation', 'channel', 'nonseed_percentile'], ascending=[True, True, False])
    .groupby(['representation', 'channel'], as_index=False).head(20))
top_candidates.to_parquet(EXPANSION / 'top_20_nonseed_candidates.parquet', index=False)
display(representation_correlations.groupby('channel').spearman_correlation.describe())
display(top_candidates.query("representation == 'all_s12'")[
    ['channel', 'permco', 'firm_name', 'propagated_score', 'nonseed_percentile']
].head(30))

## 10. Freeze the expansion outputs

The output manifest records row counts and hashes before event returns are loaded. The industry baseline remains an explicit prerequisite for the evaluation notebook.

In [12]:
artifact_rows = {
    'text_only_scores.parquet': len(text_scores),
    'embedding_propagated_scores.parquet': len(embedding_scores),
    'characteristic_propagated_scores.parquet': len(characteristic_scores),
    'shuffled_embedding_placebos_100.parquet': len(placebo_scores),
    'top_20_nonseed_candidates.parquet': len(top_candidates),
    'representation_rank_correlations.parquet': len(representation_correlations),
}
inventory_rows = []
for filename, rows in artifact_rows.items():
    artifact_path = EXPANSION / filename
    inventory_rows.append({
        'artifact': artifact_path.stem, 'rows': rows, 'path': filename,
        'bytes': artifact_path.stat().st_size,
        'sha256': hashlib.sha256(artifact_path.read_bytes()).hexdigest(),
    })
output_inventory = pd.DataFrame(inventory_rows)
output_inventory.to_parquet(EXPANSION / 'output_inventory.parquet', index=False)
expansion_manifest = {
    'stage': 'outcome_free_embedding_expansion', 'outcomes_loaded': OUTCOMES_LOADED,
    'seed_sha256': seed_manifest['seed_sha256'], 'design_sha256': design_sha256,
    'representations': sorted(embedding_frames), 'channels': channels,
    'common_firms': len(reference_firms), 'seed_firms': int(seeds.firm_id.nunique()),
    'artifacts': output_inventory.to_dict('records'),
    'industry_baseline_status': 'required_before_event_evaluation',
}
expansion_manifest['sha256'] = hashlib.sha256(
    json.dumps(expansion_manifest, sort_keys=True).encode()
).hexdigest()
(MANIFESTS / 'embedding_expansion_freeze_v1.json').write_text(json.dumps(expansion_manifest, indent=2))
display(output_inventory)
print('Expansion manifest:', expansion_manifest)
assert not OUTCOMES_LOADED

,artifact,rows,path,bytes,sha256
0,text_only_scores,5415,text_only_scores.parquet,50696,2ddc592fec2e96ec2e30a3e971c7804e7734640055d016...
1,embedding_propagated_scores,27075,embedding_propagated_scores.parquet,703345,a8dc12e36427c9fd43de65770e855a87422a6a726c5f2c...
2,characteristic_propagated_scores,5415,characteristic_propagated_scores.parquet,185802,835740a0d8426b7bc7763377447e4a807c0fb9691b5793...
3,shuffled_embedding_placebos_100,541500,shuffled_embedding_placebos_100.parquet,5372895,64dc12a2cbdd28cc3575d038710813f8eee00eef886968...
4,top_20_nonseed_candidates,300,top_20_nonseed_candidates.parquet,17107,1db8c988055541e2dfae9c891e1d89941285d4afff404e...
5,representation_rank_correlations,30,representation_rank_correlations.parquet,3396,2c56b0af3f1671621a10255eff8fe4d1dc77bc30728837...


Expansion manifest: {'stage': 'outcome_free_embedding_expansion', 'outcomes_loaded': False, 'seed_sha256': 'f159fbc59073a27619fd46c34018c440c2f506ec5be10fb086801eaeb0dc8efa', 'design_sha256': 'e632eabfec6b5f3b79e11321899af5e2bd31790b20ecb464db90475d4677e898', 'representations': ['all_s12', 'benchmark_rebalancer', 'passive_like', 'stable_stock_picker', 'tactical_active'], 'channels': ['defense_demand', 'oil_gas_production', 'russia_ukraine_operations_revenue'], 'common_firms': 1805, 'seed_firms': 9, 'artifacts': [{'artifact': 'text_only_scores', 'rows': 5415, 'path': 'text_only_scores.parquet', 'bytes': 50696, 'sha256': '2ddc592fec2e96ec2e30a3e971c7804e7734640055d0169f2da5446d486ebb61'}, {'artifact': 'embedding_propagated_scores', 'rows': 27075, 'path': 'embedding_propagated_scores.parquet', 'bytes': 703345, 'sha256': 'a8dc12e36427c9fd43de65770e855a87422a6a726c5f2c734f78c9738a065152'}, {'artifact': 'characteristic_propagated_scores', 'rows': 5415, 'path': 'characteristic_propagated_scor